In [7]:

import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import optax
import numpy as np
from pyscf import gto, scf, fci
import flax.linen as nn
from flax import nnx

# 禁用 JIT 调试（正式运行时可注释掉）
# jax.config.update('jax_disable_jit', True)

# ==============================================================================
# 1. H₂ 分子与 FCI 基准
# ==============================================================================
K = 2
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, _ = cisolver.kernel()
print("=" * 60)
print("FCI 基准能量 (Ha)")
for i, e in enumerate(E_fcis):
    print(f"E{i} = {e:.8f}  |  激发能 = {(e - E_fcis[0]) * 27.2114:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)

# ==============================================================================
# 2. 扩展希尔伯特空间与采样器
# ==============================================================================
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)
hi_ext = hi ** K

edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)
sampler = nk.sampler.MetropolisSampler(hi_ext, rule=tensor_rule, n_chains=16, sweep_size=32)

class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)
    
class TotalAnsatz(nnx.Module):
    """NES 总 Ansatz: 输入 K 个副本状态，返回 det[ψ_j(x^i)]"""
    def __init__(self, n_spin_orbitals: int, n_states: int = K, hidden_dim: int = 12):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals

        key = jax.random.key(42)
        keys = jax.random.split(key, n_states)

        self.single_ansatz_list = [
            SingleStateAnsatz(
                n_spin_orbitals,
                hidden_dim,
                rngs=nnx.Rngs(keys[i])
            )
            for i in range(n_states)
        ]

    def __call__(self, x_batch):
        """
        x_batch: (n_states, n_spin_orbitals) -> (psi_total, M)
        M_{ij} = ψ_j(x^i)
        """
        K_state = self.n_states
        M = []
        for i in range(K_state):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(K_state)]
            M.append(jnp.array(row))
        M = jnp.stack(M)
        psi_total = jnp.linalg.det(M)
        return psi_total, M


# 实例化 TotalAnsatz
total_ansatz = TotalAnsatz(
    n_spin_orbitals=4,
    n_states=K,
    hidden_dim=12,
)

graph_def, params = nnx.split(total_ansatz)

FCI 基准能量 (Ha)
E0 = -1.01546825  |  激发能 = 0.0000 eV
E1 = -0.87542794  |  激发能 = 3.8107 eV
E2 = -0.42938376  |  激发能 = 15.9482 eV
E3 = -0.26922131  |  激发能 = 20.3064 eV


In [21]:
hi_ext.all_states().shape

(16, 8)

In [23]:
hi_ext.all_states().reshape(-1,2,4).shape

(16, 2, 4)

In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx

K = 2  # 你定义的 K

class TotalAnsatz(nnx.Module):
    """NES 总 Ansatz: 输入 K 个副本状态，返回 det[ψ_j(x^i)]"""
    def __init__(self, n_spin_orbitals: int, n_states: int = K, hidden_dim: int = 12):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals

        key = jax.random.key(42)
        keys = jax.random.split(key, n_states)

        self.single_ansatz_list = [
            SingleStateAnsatz(
                n_spin_orbitals,
                hidden_dim,
                rngs=nnx.Rngs(keys[i])
            )
            for i in range(n_states)
        ]

    def forward_single(self, x_single):
        """
        处理 单个 样本！
        x_single: (K, n_spin_orbitals)
        返回：det(M), M
        完全不带 batch，纯逻辑，干净可微分
        """
        K = self.n_states
        M = []
        for i in range(K):
            row = [self.single_ansatz_list[j](x_single[i]) for j in range(K)]
            M.append(jnp.array(row))
        M = jnp.stack(M)
        det_M = jnp.linalg.det(M)
        return det_M, M

    def __call__(self, x_batch):
        """
        自动支持批处理！
        x_batch: (batch_size, K, n_spin_orbitals)
        返回：
        det_M: (batch_size,)
        M_mat: (batch_size, K, K)
        无 if，无 Python 循环，完全可JIT，可微分
        """
        # 🔥 唯一核心：用 vmap 向量化，这是 JAX 标准做法
        det_log_M, log_M_mat = jax.vmap(self.forward_single)(x_batch)
        self.current_log_M = log_M_mat
        return det_log_M

total_ansatz = TotalAnsatz(
    n_spin_orbitals=4,
    n_states=2,
    hidden_dim=12,
)

In [95]:
total_ansatz(hi_ext.all_states()[:3].reshape(-1,2,4))


(Array([ 0.        +0.j        , -1.40143019+0.29339068j,
         0.97811644-0.49861506j], dtype=complex128),
 Array([[[-1.40747088-1.27764208j,  0.37783728+0.41746251j],
         [-1.40747088-1.27764208j,  0.37783728+0.41746251j]],
 
        [[-1.40747088-1.27764208j,  0.37783728+0.41746251j],
         [ 0.38417596-1.97187624j,  0.27169049-0.03967725j]],
 
        [[-1.40747088-1.27764208j,  0.37783728+0.41746251j],
         [ 0.57695609-1.46483386j, -0.4172653 +0.95514734j]]],      dtype=complex128))

In [96]:
def create_machine(model: nnx.Module):
    """将 Flax NNX 模型包装为 NetKet 风格的 machine 函数"""
    graphdef, state = nnx.split(model)
    
    @jax.jit
    def machine(params, sigma):
        m = nnx.merge(graphdef, params)
        return m(sigma)
    
    return machine, graphdef, state

# ===================== 哈密顿量作用计算 =====================
def Ham_psi(ha, model, x):
    """计算 Hψ(x)"""
    x_primes, mels = ha.get_conn(x)
    psi_values = jax.vmap(model)(x_primes)
    H_psi_x = jnp.sum(mels * psi_values)
    return H_psi_x


def Ham_Psi(ha, total_ansatz, x):
    """计算扩展哈密顿量作用在总 Ansatz 上的矩阵"""
    k = total_ansatz.n_states
    if x.shape[0] != k:
        raise ValueError(f"Input array must have shape ({k},) but got shape {x.shape}")

    H_psi_x_i = []
    for i in range(k):
        tmp = []
        for j in range(k):
            ele = Ham_psi(ha, model=total_ansatz.single_ansatz_list[j], x=x[i])
            tmp.append(ele)
        H_psi_x_i.append(tmp)

    HPsi = jnp.array(H_psi_x_i).reshape(k, k)
    return HPsi

print("哈密顿量作用计算函数定义完成!")

哈密顿量作用计算函数定义完成!


In [81]:
machine, graphdef, state = create_machine(total_ansatz)
machine(state,x_batch) #不会报错
sampler.init_state(machine,state,12)

AttributeError: 'tuple' object has no attribute 'dtype'

In [72]:
machine(state,x_batch)
        

(Array([ 0.        +0.j        , -1.40143019+0.29339068j,
         0.97811644-0.49861506j,  0.3814984 +0.74532574j,
         1.40143019-0.29339068j,  0.        +0.j        ,
         1.62449615+1.61061363j,  0.41862447+1.46444159j,
        -0.97811644+0.49861506j, -1.62449615-1.61061363j,
         0.        +0.j        , -1.26211218+0.16502531j,
        -0.3814984 -0.74532574j, -0.41862447-1.46444159j,
         1.26211218-0.16502531j,  0.        +0.j        ],      dtype=complex128),
 Array([[[-1.40747088-1.27764208j,  0.37783728+0.41746251j],
         [-1.40747088-1.27764208j,  0.37783728+0.41746251j]],
 
        [[-1.40747088-1.27764208j,  0.37783728+0.41746251j],
         [ 0.38417596-1.97187624j,  0.27169049-0.03967725j]],
 
        [[-1.40747088-1.27764208j,  0.37783728+0.41746251j],
         [ 0.57695609-1.46483386j, -0.4172653 +0.95514734j]],
 
        [[-1.40747088-1.27764208j,  0.37783728+0.41746251j],
         [ 0.07622913-1.50397417j, -0.47824368+0.28571334j]],
 
        [[ 